In [ ]:
"""
Animal Classification with Continuous Attribute Learning (CAL)
This module implements a deep learning model that combines traditional image classification
with continuous attribute learning for improved accuracy and interpretability.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# Enable GPU optimizations for faster training
torch.backends.cudnn.benchmark = True

# Model and training configuration parameters
CONFIG = {
    'data_path': '/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/train',
    'batch_size': 32,          # Number of images per batch
    'num_epochs': 50,          # Total training epochs
    'learning_rate': 0.001,    # Initial learning rate
    'num_attributes': 85,      # Number of continuous attributes to predict
    'num_classes': 50,         # Number of animal classes
    'smoothing': 0.1,          # Label smoothing factor
    'weight_decay': 0.01,      # L2 regularization factor
    'image_size': 224         # Input image size for the model
}

class LabelSmoothingCrossEntropy(nn.Module):
    """
    Implements Label Smoothing Cross Entropy Loss.
    This loss function helps prevent the model from becoming overconfident
    and improves generalization.
    
    Args:
        smoothing (float): Smoothing factor, typically small (e.g., 0.1)
    """
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        
    def forward(self, x, target):
        confidence = 1. - self.smoothing
        logprobs = F.log_softmax(x, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        loss = confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()

def load_data():
    """
    Loads and preprocesses the dataset.
    
    Returns:
        tuple: (class_mapping, continuous_attributes)
            - class_mapping: Dictionary mapping class names to indices
            - continuous_attributes: Normalized attribute matrix
    """
    # Read class mapping from file
    class_df = pd.read_csv('/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/classes.txt', 
                          delimiter='\s+', header=None, names=['index', 'class_name'])
    class_mapping = {row['class_name']: row['index']-1 for _, row in class_df.iterrows()}
    
    # Load and normalize continuous attribute matrices
    continuous_attributes = pd.read_csv('/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/predicate-matrix-continuous.txt', 
                                      delimiter='\s+', header=None).values.astype(np.float32)
    continuous_attributes = (continuous_attributes - continuous_attributes.min()) / (continuous_attributes.max() - continuous_attributes.min())
    
    return class_mapping, continuous_attributes

class AnimalAttributeDataset(Dataset):
    """
    Custom Dataset class for loading animal images with their attributes and labels.
    
    Args:
        image_paths (list): List of paths to image files
        labels (list): List of class labels corresponding to images
        attributes (np.array): Matrix of continuous attributes for each class
        transform (callable, optional): Optional transform to be applied to images
    """
    def __init__(self, image_paths, labels, attributes, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.attributes = attributes
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform:
                image = self.transform(image)
            
            label = self.labels[idx]
            attribute = self.attributes[label]
            
            return image, torch.tensor(label, dtype=torch.long), torch.tensor(attribute, dtype=torch.float32)
        except Exception as e:
            print(f"Error loading image {self.image_paths[idx]}: {str(e)}")
            return None

def collate_fn(batch):
    """
    Custom collate function to handle None values in batch.
    Filters out None values that might occur due to corrupted images.
    """
    batch = list(filter(lambda x: x is not None, batch))
    if len(batch) == 0:
        raise RuntimeError("Batch is empty after filtering")
    return torch.utils.data.dataloader.default_collate(batch)

def prepare_data(class_mapping, continuous_attributes):
    """
    Prepares data loaders for training and validation.
    
    Args:
        class_mapping (dict): Mapping from class names to indices
        continuous_attributes (np.array): Matrix of continuous attributes
        
    Returns:
        tuple: (train_loader, val_loader)
    """
    image_paths = []
    labels = []
    
    # Walk through the dataset directory and collect image paths and labels
    for class_name in os.listdir(CONFIG['data_path']):
        if class_name in class_mapping:
            class_path = os.path.join(CONFIG['data_path'], class_name)
            if os.path.isdir(class_path):
                class_label = class_mapping[class_name]
                for img_name in os.listdir(class_path):
                    if img_name.endswith(('.jpg', '.jpeg', '.png')):
                        img_path = os.path.join(class_path, img_name)
                        image_paths.append(img_path)
                        labels.append(class_label)
    
    print(f"Total images found: {len(image_paths)}")
    print(f"Unique labels: {sorted(set(labels))}")
    
    # Split data into training and validation sets
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        image_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    # Define training data augmentation pipeline
    train_transform = transforms.Compose([
        transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Define validation data transforms (no augmentation)
    val_transform = transforms.Compose([
        transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = AnimalAttributeDataset(train_paths, train_labels, continuous_attributes, train_transform)
    val_dataset = AnimalAttributeDataset(val_paths, val_labels, continuous_attributes, val_transform)
    
    # Create and return data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    return train_loader, val_loader

class CALModel(nn.Module):
    """
    Continuous Attribute Learning Model for animal classification.
    
    Architecture:
    1. ResNet50 backbone for feature extraction
    2. Attribute prediction head (features -> continuous attributes)
    3. Class prediction head (attributes -> class probabilities)
    
    Args:
        num_attributes (int): Number of continuous attributes to predict
        num_classes (int): Number of output classes
    """
    def __init__(self, num_attributes, num_classes):
        super(CALModel, self).__init__()
        
        # Load pretrained ResNet backbone
        self.backbone = models.resnet50(pretrained=True)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Attribute prediction head
        self.attribute_predictor = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.BatchNorm1d(512),
            nn.Linear(512, num_attributes),
            nn.Sigmoid()
        )
        
        # Class prediction head
        self.class_predictor = nn.Sequential(
            nn.Linear(num_attributes, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.BatchNorm1d(512),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        attributes = self.attribute_predictor(features)
        classes = self.class_predictor(attributes)
        return attributes, classes

def train_model():
    """
    Main training loop for the CAL model.
    
    Features:
    - Dynamic loss weighting between attribute and classification tasks
    - One-cycle learning rate scheduling
    - Gradient clipping for stability
    - Model checkpointing based on validation accuracy
    
    Returns:
        None (saves best model checkpoint to disk)
    """
    # Setup device (GPU if available, else CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load and prepare data
    class_mapping, continuous_attributes = load_data()
    train_loader, val_loader = prepare_data(class_mapping, continuous_attributes)
    
    # Initialize model
    model = CALModel(CONFIG['num_attributes'], CONFIG['num_classes']).to(device)
    
    # Define loss functions
    criterion_attr = nn.BCELoss()
    criterion_cls = LabelSmoothingCrossEntropy(smoothing=CONFIG['smoothing'])
    
    # Setup optimizer with different learning rates for different components
    params = [
        {'params': model.backbone.parameters(), 'lr': CONFIG['learning_rate'] * 0.1},
        {'params': model.attribute_predictor.parameters(), 'lr': CONFIG['learning_rate']},
        {'params': model.class_predictor.parameters(), 'lr': CONFIG['learning_rate']}
    ]
    
    optimizer = optim.AdamW(params, weight_decay=CONFIG['weight_decay'])
    
    # Setup learning rate scheduler
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[CONFIG['learning_rate'] * 0.1, CONFIG['learning_rate'], CONFIG['learning_rate']],
        epochs=CONFIG['num_epochs'],
        steps_per_epoch=len(train_loader)
    )
    
    best_val_acc = 0
    
    # Training loop
    for epoch in range(CONFIG['num_epochs']):
        # Training phase
        model.train()
        train_loss = 0
        train_attr_loss = 0
        train_cls_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{CONFIG["num_epochs"]}')
        for images, labels, attributes in pbar:
            images = images.to(device)
            labels = labels.to(device)
            attributes = attributes.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            pred_attributes, pred_classes = model(images)
            
            # Calculate losses
            loss_attr = criterion_attr(pred_attributes, attributes)
            loss_cls = criterion_cls(pred_classes, labels)
            
            # Dynamic loss weighting
            attr_weight = max(0.3, 0.7 - (epoch / CONFIG['num_epochs']) * 0.4)
            loss = (attr_weight * loss_attr) + ((1 - attr_weight) * loss_cls)
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            # Update metrics
            train_loss += loss.item()
            train_attr_loss += loss_attr.item()
            train_cls_loss += loss_cls.item()
            
            _, predicted = torch.max(pred_classes.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({
                'loss': train_loss/(pbar.n+1),
                'acc': 100.*correct/total,
                'attr_w': attr_weight
            })
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        val_attr_accuracy = 0
        
        with torch.no_grad():
            for images, labels, attributes in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                attributes = attributes.to(device)
                
                pred_attributes, pred_classes = model(images)
                
                loss_attr = criterion_attr(pred_attributes, attributes)
                loss_cls = criterion_cls(pred_classes, labels)
                loss = (attr_weight * loss_attr) + ((1 - attr_weight) * loss_cls)
                
                val_loss += loss.item()
                
                _, predicted = torch.max(pred_classes.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                attr_pred = (pred_attributes > 0.5).float()
                val_attr_accuracy += (attr_pred == attributes).float().mean().item()
        
        # Calculate validation metrics
        val_accuracy = 100 * val_correct / val_total
        avg_attr_accuracy = 100 * val_attr_accuracy / len(val_loader)
        
        # Print epoch results
        print(f'\nEpoch {epoch+1}/{CONFIG["num_epochs"]}:')
        print(f'Train Loss: {train_loss/len(train_loader):.4f}')
        print(f'Train Class Loss: {train_cls_loss/len(train_loader):.4f}')
        print(f'Train Attr Loss: {train_attr_loss/len(train_loader):.4f}')
        print(f'Val Loss: {val_loss/len(val_loader):.4f}')
        print(f'Val Class Accuracy: {val_accuracy:.2f}%')
        print(f'Val Attribute Accuracy: {avg_attr_accuracy:.2f}%')
        
        # Save best model based on validation accuracy
        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_accuracy,
            }, 'best_model.pth')

In [ ]:
if __name__ == "__main__":
    train_model()